# 🤖 Robot Maintenance Data EDA

This notebook analyzes robot axis current data to identify patterns useful for predictive maintenance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os

# Ensure directories
os.makedirs('../data/processed', exist_ok=True)

**📝 Explanation:**
We check the environment imports. Key libraries include `pandas` for handling the robot CSV data and `matplotlib` for visualizing the current waveforms.

In [ ]:
# Load Raw Data
df = pd.read_csv('../data/raw/robot_maintenance_data.csv')

# Convert Time to Datetime
df['Time'] = pd.to_datetime(df['Time'])

# Filter for 'current' trait only
df_current = df[df['Trait'] == 'current'].copy()

# Sort by Time
df_current = df_current.sort_values('Time')

# Fill NaNs with 0 (assuming sensor dropout means 0 current) or forward fill
df_current = df_current.fillna(0)

# Save processed version to CSV
df_current.to_csv('../data/processed/robot_current_clean.csv', index=False)

print(f"Data Shape: {df_current.shape}")
df_current.head()

**📝 Explanation:**
We load the raw maintenance data. We also parse the `Time` column to datetime objects to enable time-series analysis and filter specifically for `current` traits, as current spikes are key indicators of load.

In [ ]:
# Save to SQLite Database (Data Engineering Requirement)
db_path = '../data/robot_data.db'
conn = sqlite3.connect(db_path)
df_current.to_sql('robot_current', conn, if_exists='replace', index=False)
conn.close()
print(f"Data saved to SQLite database at {db_path}")

**📝 Explanation:**
We persist the cleaned data into a SQLite database (`robot_data.db`). This demonstrates a standard Data Engineering capability: transforming raw logs into a structured, queryable format.

In [ ]:
# Visualize Axis 1 Current over Time
plt.figure(figsize=(12, 6))
plt.plot(df_current['Time'], df_current['Axis #1'], label='Axis #1 Current', alpha=0.7)
plt.title('Axis #1 Current Over Time')
plt.xlabel('Time')
plt.ylabel('Current (A)')
plt.legend()
plt.grid(True)
plt.show()

**📝 Explanation:**
We visualize `Axis #1` current over time. This helps us see the duty cycle of the robot and identify any obvious outliers or periods of inactivity.

In [ ]:
# Correlation Matrix
# We ignore 'Trait' and 'Time' for correlation
axis_cols = [c for c in df_current.columns if 'Axis' in c]
corr = df_current[axis_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title('Correlation between Robot Axes Currents')
plt.show()

# Identify highest correlation pair for Regression
corr_unstacked = corr.abs().unstack()
corr_sorted = corr_unstacked[corr_unstacked < 1.0].sort_values(ascending=False)
print("Top Correlations:")
print(corr_sorted.head(5))

**📝 Explanation:**
We generate a correlation matrix between all axes. High correlation suggests axes moving together (synergy), which is useful for selecting features for our regression model.